Suppose we model a population of bacteria whose rate of growth is proportional to its current size: $$\frac{dS}{dt} = \mu S$$ Solving this, we would get: $S_t = S_0e^{\mu t}$. However, stock prices do not increase exponentially, they jitter unpredictably about the growth trend, therefore we must introduce an element of randomness to the equation using a mathematical representation of Brownian motion. Let us define some term $W_t$ that has the following properties:
1. $W_0 = 0$
2. Every increment of $W$ is independant: for every $t\gt 0$ the future increments, $W_{t_u}-W_t\gt 0 $ are independant of the past values $W_s, \forall s\lt t$
3. $W$ has Gaussian (normally distributed) increments: $\forall u,t\ge 0 , W_{t+u}-W_t\sim \mathcal{N}(0,u)$. Note that the variance of the increment is dependant on the size of the increment.
4. The function $t\mapsto W_t$ is continuous

This is known as a **Wiener Process** and it will allow us to simulate the "jitters" and random motion observed in real stocks. We can do this by assuming the same upward drift, but introducing our Wiener process like so: $$dS = \mu S \: dt + \sigma S \:dW$$ Here $\sigma$ is the **volatility** of the stock, it acts as a scale factor so the magnitude of the random motions can be emphasized by more volatile stocks and suppressed by less volatile ones. Also note that the random motion is also scaled by the current stock price. This emulates real life more closely as it allows the equation to vary the stock by percentage rather than by an absolute price change (a $£1$ price swing is very different in a $£5$ stock vs a $£500$ stock, this is also why the resulting solution is in the form of a lognormal dirstibution rather than a normal one). 

Now before we proceed, we have to consider Itô's Lemma. Lets say you take some function $V(S,t)$ which represents the value of the option, and it takes the stock price $S$ and time $t$ as arguments. Now let us attempt to find the rate of change of V in relation to a changing $S$ or a changing $t$. If we attempt to solve this using the ordinary multivariable chain rule we get: $$dV = \frac{\partial V}{\partial t} dt +\frac{\partial V}{\partial S} dS$$ However, due to our Wiener process, this ordinary chain rule cannot be used (the $dW^n, n\gt 2$ terms do not vanish as it's increments are distributed normally), in its place we use Itô's Lemma, which adds an extra term to account for the random variable: $$dV = \frac{\partial V}{\partial t}dt+\frac{\partial V}{\partial S}dS+\frac{1}{2}\frac{\partial^2 V}{\partial S^2}(dS)^2$$ Now considering $dS$ from our SDE we get: $$\begin{aligned}(dS)^2 &= \underbrace{\mu^2S^2(dt)^2+2\sigma\mu S^2 \:dt \:dW}_{\text{Negligible}}+\sigma^2S^2(dW)^2\\&\approx\sigma ^2 S^2 \:dt\end{aligned}$$ The approximation holds as the first term is a factor of $dt^2$ which is negligible, however showing this for the second term is not as simple. We can find the mean of $dW$ using the moment generating function for $dW$:$$\begin{aligned}&M_X(x) = \exp\left(\mu t + \frac{\sigma^2x^2}{2}\right) = \exp{\left(\frac{dt\:x^2}{2}\right)} \\ \implies &M_X^{(1)}(x) = x\:dt\:\exp{\left(\frac{dt\:x^2}{2}\right)} \\\implies &M_X^{(2)}(x) = dt\:\exp{\left(\frac{dt\:x^2}{2}\right)}+x^2(dt)^2\exp{\left(\frac{dt\:x^2}{2}\right)}\\\implies &E[(dW)^2] = M_X^{(2)}(0) = dt \end{aligned}$$ Therefore, $(dW)^2$ is on the order of $dt$ and looking at the variance (using the standard rule) we get $\text{Var}((dW)^2) = 2(dt)^2$. Why is this useful? Lets consider this distribution over the interval $[0,T]$ and lets sum the variance and mean over $n = \frac{T}{dt}$ independant subintervals: $$\begin{gathered}E\left[\sum_i(dW_i)^2\right] = n\cdot dt = T\\ \text{Var}\left[\sum_i(dW_i)^2\right] = n\cdot 2(dt)^2 = 2T\cdot dt\end{gathered}$$ Note that the variance tends to 0 as $dt$ tends to 0 (as should be expected by the Law of Large Numbers). This is what permits us to use $(dW)^2 = \sigma ^2 S^2$ in Itô's Lemma. Now let us substitute our SDE and the approximation we just derived into Itô's Lemma to get:$$dV = \left(\frac{\partial V}{\partial t} + \mu S\frac{\partial V}{\partial S}+\frac{1}{2}\sigma^2S^2\frac{\partial^2 V}{\partial S^2}\right)dt + \sigma S\frac{\partial V}{\partial S}dW$$

We can then solve this SDE (stochastic differential equation. Let us set $V(S,t) = \ln S$ $$dV = \left(\mu -\frac{\sigma^2}{2}\right)dt +\sigma \:dW $$ Note that we no longer have any $S$ terms on the right hand side so we can now simply integrate across $[0,T]$ to get $$\ln S_T-\ln S_0 = \left(\mu -\frac{\sigma^2}{2}\right)T+\sigma W_T$$ We know that $W_T\sim \mathcal{N}(0, T)$. This can be rewritten by scaling the standard normal distribution $Z\sim \mathcal{N}(0,1)$ by the standard deviation of $W_T$ which is $\sqrt{T}$:$$ \therefore S_T = S_0\exp\left[\left(\mu-\frac{\sigma^2}{2}\right)T+\sigma Z\sqrt{T} \right] \quad Z\sim\mathcal{N}(0,1)$$ Let's pause here for a moment and figure out what we have calculated. We have found an analytical equation that returns a prediction for the value of a stock $S_T$ after a time $T$ from its initial value $S_0$, its volatility $\sigma$ and $\mu$. But what is $\mu$? We used it at the very beginning without really saying what it is. When we initiated it, it was the expected rate of growth of a population as a percentage of its current size, in terms of stocks, this is the expected rate of return from this specific stock on average per unit time. This value is very subjective and is, in essence, unobservable. Not very useful for our use case. To counteract this and get something that we can use we have to subsitute this value for something that is more universal.

To do this we have to **hedge** the option with something else. Let us build a portfolio as follows. Let us hold a call option of value $V$ (note this is *not* the $\ln S$ we ascribed to $V$ earlier) and short $\Delta$ shares *of the same stock*. We can represent the portfolio like so: $$\Pi = V-\Delta S$$ Applying Itô's Lemma, we can see how $\Pi$ changes over a small time step: $$d\Pi = dV-\Delta dS$$ Substituting Itô's Lemma and the SDE for $dV, dS$ respectively: $$\begin{aligned}&d\Pi = \left(\frac{\partial V}{\partial t} + \mu S\frac{\partial V}{\partial S}+\frac{1}{2}\sigma^2S^2\frac{\partial^2 V}{\partial S^2}\right)dt + \sigma S\frac{\partial V}{\partial S}dW - \Delta(\mu S\:dt+\sigma S\:dW)\\\implies
& d\Pi = \left(\frac{\partial V}{\partial t} + \mu S\left(\frac{\partial V}{\partial S}-\Delta\right)+\frac{1}{2}\sigma^2S^2\frac{\partial^2 V}{\partial S^2}\right)dt + \left(\sigma S\left(\frac{\partial V}{\partial S}-\Delta\right)\right)dW
\end{aligned}$$ We can then set $\Delta = \frac{\partial V}{\partial S}$ to get: $$d\Pi = \left(\frac{\partial V}{\partial t}+\frac{1}{2}\sigma^2S^2\frac{\partial^2 V}{\partial S^2}\right)dt$$ Note that what we have now is an entirely deterministic equation, **the random term has vanished**. We have made a portfolio that must grow at a constant rate. We have manufactured a **risk-free** portfolio. Therefore, by the **no-arbitrage** principle it must grow at the **risk-free rate** ($r$): $$\frac{d\Pi}{dt} = \frac{\partial V}{\partial t}+\frac{1}{2}\sigma^2S^2\frac{\partial^2 V}{\partial S^2} = r\Pi = r\left(V-S\frac{\partial V}{\partial S}\right)$$ Rearranging this we get the **Black-Scholes PDE**: $$\boxed{\frac{\partial V}{\partial t} +\frac{1}{2}\sigma^2S^2\frac{\partial^2V}{\partial S^2}+rS\frac{\partial V}{\partial S}-rV = 0}$$